In [ ]:
import glob, pathlib, json, sys

sys.path.append(r"D:\BEHAVIOR-1K\asset_pipeline")
import b1k_pipeline.utils

In [ ]:
provided = set()
no_objects = []
for target in b1k_pipeline.utils.get_targets("objects"):
    if not target.startswith("objects/legacy"):
        continue

    j_path = (
        b1k_pipeline.utils.PIPELINE_ROOT / "cad" / target / "artifacts/object_list.json"
    )
    with open(j_path, "r") as f:
        object_list = json.load(f)

    if len(object_list["provided_objects"]) == 0:
        no_objects.append(target)

    provided.update(set(object_list["provided_objects"]))

print("These targets provide no objects:")
print(no_objects)

In [ ]:
provided

In [ ]:
# What objects are missing from the model rename list
import yaml

with open(r"D:\BEHAVIOR-1K\asset_pipeline\b1k_pipeline\model_rename.yml", "r") as f:
    translation_dict = yaml.load(f, yaml.SafeLoader)
    expected = set(x.replace("/", "-") for x in translation_dict.values())
missing = sorted(expected - provided)
print(missing)

In [ ]:
# For each of the above, what is their file exporting
for m in missing:
    target = f"objects/legacy_{m}"
    j_path = (
        b1k_pipeline.utils.PIPELINE_ROOT / "cad" / target / "artifacts/object_list.json"
    )
    if not j_path.exists():
        print(f"{m} file is missing.")
        continue

    with open(j_path, "r") as f:
        object_list = json.load(f)

    if len(object_list["provided_objects"]) == 0:
        print(f"{m} file provides no objects")
    else:
        other = ", ".join(object_list["provided_objects"])
        print(f"{m} file provides other objects: {other}")

In [ ]:
import sys, os

sys.path.append(r"D:\BEHAVIOR-1K\asset_pipeline")
from b1k_pipeline.urdfpy import URDF

import numpy as np
import tqdm

# Check the joints from the original files
problem_objects = {}
problem_joint_limits = []
for k, v in tqdm.tqdm(translation_dict.items()):
    old_category_name, old_model_name = k.split("/")
    model_dir = os.path.join(
        r"C:\Users\Cem\research\iGibson-dev\igibson\data\ig_dataset",
        "objects",
        old_category_name,
        old_model_name,
    )

    # Load the URDF file into urdfpy
    urdf_filename = old_model_name + ".urdf"
    urdf_path = os.path.join(model_dir, urdf_filename)
    robot = URDF.load(urdf_path)

    # Check joint limits
    problem_joints = [
        j
        for j in robot.joints
        if j.joint_type == "revolute"
        and np.abs(np.rad2deg(j.limit.upper - j.limit.lower)) >= 179
    ]
    problem_joint_limits.extend(
        [
            np.abs(np.rad2deg(joint.limit.upper - joint.limit.lower))
            for joint in robot.joints
            if joint.joint_type == "revolute"
        ]
    )
    if problem_joints:
        problem_objects[v] = [j.child for j in problem_joints]

In [ ]:
print(problem_objects)

In [ ]:
print(len(problem_objects))

In [ ]:
print(len(problem_joint_limits))

In [ ]:
import matplotlib.pyplot as plt

x = np.array(problem_joint_limits)
plt.hist(x[x >= 179], bins=10)
plt.show()
print(len(x[x >= 181]))

In [ ]:
# Check that the corresponding file exists for all
targets = []
for po_name, joints in problem_objects.items():
    obj_name = po_name.replace("/", "-")
    target = "objects/legacy_" + obj_name
    targets.append(target)

    filename = b1k_pipeline.utils.PIPELINE_ROOT / "cad" / target
    assert filename.exists(), str(filename)
    print(filename, "exists")

    # Get the corresponding object list
    obj_list = filename / "artifacts/object_list.json"
    assert obj_list.exists(), "No obj list for " + str(filename)
    with open(obj_list, "r") as f:
        meshes = json.load(f)["meshes"]

    joint_failures = []
    for joint in joints:
        # Check that some link with this name exists
        this_link_name = f"{obj_name}-0-{joint}".lower()
        link_meshes = [
            x
            for x in meshes
            if b1k_pipeline.utils.parse_name(x)["link_basename"] == this_link_name
        ]
        if not link_meshes:
            joint_failures.append(this_link_name)
    if joint_failures:
        print(f"Could not find joints {joint_failures}. Mesh list: {meshes}")

In [ ]:
print("dvc unprotect", "\n".join("cad/" + x + "/processed.max" for x in targets))

In [ ]:
print("\n".join('"' + x.replace("/", "-") + '",' for x in problem_objects.keys()))